# 00 · Configuration, Contracts & Database

Run this first. The standalone database uses SQLite; PostgreSQL/pgvector is optional in notebook 02. Synthetic content is never labeled as verified live research.

In [ ]:
from __future__ import annotations
import io, os, re, json, csv, hashlib, sqlite3, mimetypes, zipfile, uuid
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter
from typing import Literal, Any
from pydantic import BaseModel, Field

ROOT = Path.cwd().parent.resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
STORE = ROOT / '.runtime'
STORE.mkdir(exist_ok=True)
DB_FILE = STORE / 'group2.sqlite3'
PROJECT_ID = '77777777-7777-4777-8777-777777777777'
DOMAINS = ('technology','clinical','sec','csr')
MAX_UPLOAD = 20 * 1024 * 1024
NOW = lambda: datetime.now(timezone.utc).isoformat()

def ensure_project_id(value):
    return str(uuid.UUID(str(value)))

def db_connect():
    cx = sqlite3.connect(DB_FILE, timeout=30)
    cx.row_factory = sqlite3.Row
    cx.execute('PRAGMA foreign_keys=ON')
    cx.execute('PRAGMA busy_timeout=30000')
    return cx

def init_db():
    with db_connect() as cx:
        cx.executescript('''
        CREATE TABLE IF NOT EXISTS documents(
            id TEXT PRIMARY KEY, project_id TEXT NOT NULL, domain TEXT NOT NULL,
            filename TEXT NOT NULL, sha256 TEXT NOT NULL, storage_key TEXT NOT NULL,
            mime_type TEXT, source_type TEXT NOT NULL DEFAULT 'uploaded',
            source_url TEXT, size_bytes INTEGER NOT NULL,
            status TEXT NOT NULL, created_at TEXT NOT NULL,
            UNIQUE(project_id,domain,sha256)
        );
        CREATE TABLE IF NOT EXISTS chunks(
            id TEXT PRIMARY KEY, document_id TEXT NOT NULL REFERENCES documents(id) ON DELETE CASCADE,
            project_id TEXT NOT NULL, domain TEXT NOT NULL, page_number INTEGER,
            chunk_index INTEGER NOT NULL, content TEXT NOT NULL,
            metadata_json TEXT NOT NULL DEFAULT '{}',
            UNIQUE(document_id,chunk_index)
        );
        CREATE INDEX IF NOT EXISTS ix_chunks_scope ON chunks(project_id,domain);
        CREATE TABLE IF NOT EXISTS evidence(
            id TEXT PRIMARY KEY, project_id TEXT NOT NULL, domain TEXT NOT NULL,
            chunk_id TEXT REFERENCES chunks(id) ON DELETE SET NULL,
            source_type TEXT NOT NULL, claim TEXT NOT NULL,
            source_url TEXT, metadata_json TEXT NOT NULL DEFAULT '{}',
            created_at TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS ix_evidence_scope ON evidence(project_id,domain);
        CREATE TABLE IF NOT EXISTS research_sources(
            id TEXT PRIMARY KEY, project_id TEXT NOT NULL, domain TEXT NOT NULL,
            url TEXT, title TEXT, publisher TEXT, publication_date TEXT,
            source_type TEXT NOT NULL, status TEXT NOT NULL, retrieved_at TEXT NOT NULL,
            metadata_json TEXT NOT NULL DEFAULT '{}'
        );
        CREATE TABLE IF NOT EXISTS research_tasks(
            id TEXT PRIMARY KEY, project_id TEXT NOT NULL, domain TEXT NOT NULL,
            question TEXT NOT NULL, source_mode TEXT NOT NULL,
            status TEXT NOT NULL, result_json TEXT NOT NULL DEFAULT '{}',
            created_at TEXT NOT NULL, completed_at TEXT
        );
        CREATE INDEX IF NOT EXISTS ix_research_tasks_scope ON research_tasks(project_id,domain);
        CREATE TABLE IF NOT EXISTS analysis_runs(
            id TEXT PRIMARY KEY, project_id TEXT NOT NULL, method TEXT NOT NULL,
            input_hash TEXT NOT NULL, result_json TEXT NOT NULL, created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS recommendations(
            id TEXT PRIMARY KEY, project_id TEXT NOT NULL, domain TEXT NOT NULL,
            query TEXT NOT NULL, candidate_id TEXT NOT NULL,
            score REAL NOT NULL, evidence_json TEXT NOT NULL, created_at TEXT NOT NULL
        );
        ''')
    return DB_FILE

class SearchInput(BaseModel):
    project_id: uuid.UUID
    question: str = Field(min_length=3,max_length=1800)
    domain: Literal['technology','clinical','sec','csr'] | None = None
    top_k: int = Field(default=5,ge=1,le=30)

class EvidenceHit(BaseModel):
    chunk_id: str
    document_id: str
    domain: str
    filename: str
    page_number: int | None
    excerpt: str
    citation: str
    source_url: str | None
    source_type: str
    score: float
    retrieval_method: str = 'hybrid-tfidf-bm25'

class CandidateInput(BaseModel):
    candidate_id: str
    name: str
    description: str = ''
    kind: str = 'action'
    compatible_models: list[str] = Field(default_factory=list)
    required_terms: list[str] = Field(default_factory=list)
    identifying_terms: list[str] = Field(default_factory=list)

class RecommendationInput(BaseModel):
    project_id: uuid.UUID
    query: str
    domain: Literal['technology','clinical','sec','csr']
    device_model: str | None = None
    candidates: list[CandidateInput]
    top_k: int = Field(default=5,ge=1,le=20)

init_db()
print('Database initialized:',DB_FILE)